# 10 · Silver: território (S1)

**Estado: completo para as fontes recebidas (11/11/2026).** Primeiro separa Recife da malha IBGE 2025 e compara ao conjunto de bairros; depois gera bairros e chaves, logradouros, água, atributos das faces e a grade H3 resolução 9. Medidas em EPSG:31985; nenhum bairro é recortado.

**Papel no Canvas:** referência territorial comum (problema 3: dados que não conversam). Beneficia o motor e as duas telas; pertence ao primeiro fluxo.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "src" / "wastestack").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import geopandas as gpd
from wastestack.config import BRONZE, SILVER
from wastestack.espacial import preparar_contorno

import pandas as pd


## 1. Fontes Bronze preservadas

IBGE: malha municipal edição 2025, SIRGAS 2000 (EPSG:4674). Prefeitura: bairros do conjunto Área urbana, CRS84/WGS84 (lido como EPSG:4326). A data de download não é o período retratado pela fonte.

As notas metodológicas do IBGE foram localizadas na página oficial, mas a consulta integral respondeu 403 nesta sessão; a consulta integral segue registrada no roadmap. Esta é uma comparação técnica das geometrias recebidas, sem conclusão sobre mudanças legais de limite.


In [2]:
ARQUIVO_IBGE = BRONZE / "limite_municipal" / "2026-11-11" / "PE_Municipios_2025.zip"
ARQUIVO_BAIRROS = BRONZE / "area_urbana" / "2026-11-11" / "bairros-do-recife.geojson"

resumo = preparar_contorno(ARQUIVO_IBGE, ARQUIVO_BAIRROS, SILVER)
resumo


{
 "executado_em": "2026-11-12T13:05:05.484157+00:00",
 "status": "contorno separado e comparação executada; notebook 10 parcial",
 "municipio_codigo": "2611606",
 "municipio_nome": "Recife",
 "municipios_na_fonte": 185,
 "bairros": 94,
 "geometrias_originais_validas": true,
 "geometrias_corrigidas": 0,
 "codigos_bairro_unicos": true,
 "crs_fontes": {
  "ibge": "EPSG:4674",
  "bairros": "EPSG:4326"
 },
 "epsg_medidas": 31985,
 "area_atributo_ibge_km2": 218.843,
 "area_ibge_utm_km2": 218.9138082056109,
 "area_uniao_bairros_utm_km2": 220.05205819914528,
 "soma_menos_uniao_bairros_m2": 0.02646377682685852,
 "area_comum_km2": 218.4682312131108,
 "ibge_sem_bairro_km2": 0.4455769924995397,
 "bairros_fora_ibge_km2": 1.5838269860349778,
 "pct_ibge_coberto_bairros": 99.79646007889937,
 "pct_bairros_no_ibge": 99.2802489560897,
 "fontes": [
  {
   "arquivo": "PE_Municipios_2025.zip",
   "sha256": "534bc4d7a146a37c579d688bf75f5b19250070531d675c575a40943ede0d2cb8"
  },
  {
   "arquivo": "bairros-do

## 2. Diferenças preservadas

`ibge_sem_bairro` contém a área do município que não está na união dos bairros. `bairro_fora_ibge` contém a situação inversa. Uma fonte não foi alterada para coincidir com a outra. A causa das diferenças não está comprovada.

O campo de área original do IBGE fica separado da medida em UTM; valores calculados por métodos diferentes não precisam coincidir. As diferenças nas bordas precisam ser consideradas antes de fixar regras de cobertura da grade ou rejeição de coordenadas.


In [3]:
cobertura = gpd.read_parquet(SILVER / "cobertura_bairro_contorno.parquet")
cobertura.nlargest(10, "area_fora_ibge_km2")[["bairro_codigo", "bairro_nome", "area_bairro_km2", "area_fora_ibge_km2", "pct_area_fora_ibge"]]


    bairro_codigo            bairro_nome  area_bairro_km2  area_fora_ibge_km2  pct_area_fora_ibge
79            205             Boa Viagem         8.338925            0.518396            6.216578
9              19                 Recife         3.140027            0.369047           11.752999
68            256       Brasília Teimosa         1.466766            0.260369           17.751209
15            418              Guabiraba        46.063553            0.205773            0.446715
4             736                 Várzea        22.391160            0.111772            0.499177
93            884  Cohab - Ibura de Cima         4.354247            0.025774            0.591918
69            248                   Pina         6.632741            0.023939            0.360919
0             752                 Curado         8.255774            0.013000            0.157466
42            221                 Jordão         1.562009            0.012158            0.778351
60            744   

In [4]:
assert resumo["municipio_codigo"] == "2611606"
assert resumo["geometrias_originais_validas"]
assert resumo["codigos_bairro_unicos"]
assert resumo["epsg_medidas"] == 31985
print("Contorno separado; comparação e releitura conferidas.")

Contorno separado; comparação e releitura conferidas.


## 3. Demais entidades e grade H3

`preparar_territorio()` (em `src/wastestack/territorio.py`) refaz o contorno e grava:

- `territorio_bairro` (94 bairros, chave `bairro_codigo`, RPA pelo campo do bairro, conferida com o CSV de bairros e RPAs);
- `malha_logradouros` e `territorio_logradouro` (rua × bairro pela relação oficial; a linha é recortada ao bairro). **Ruas de borda:** quando a relação existe mas a linha corre fora do polígono, usa-se o trecho a até 30 m do limite, com o método gravado;
- `linhas_agua_logradouro`: os 14 rios cadastrados como rua (nome original termina em "-RIO"/"-RIACHO", mais o Canal do Jordão) saem da malha viária e ficam como linhas de água. Ruas como "RUA BEIRA RIO" continuam ruas. "RUA LIMITE DE MUNICIPIO" sai;
- `territorio_agua` (classes; só rio, canal e riacho servem para distância);
- `face_quadra`: atributos decodificados, **sem geometria**. A ligação é rua + bairro; a posição exata da face não é inventada. V0 = 0 vira nulo;
- `grade_h3` e `grade_h3_bairro`. **Política de borda:** o domínio é a união IBGE ∪ bairros; hexágonos inteiros; o bairro de referência é o de maior interseção; `borda_divergente` marca as células nas áreas em que as fontes discordam. Nada é descartado em silêncio.

In [5]:
from wastestack.territorio import preparar_territorio
estatisticas = preparar_territorio()
estatisticas

{
 "bairros": 94,
 "relacoes_rua_bairro": 11921,
 "relacoes_sem_linha_util": 223,
 "relacoes_ambiguas": 127,
 "rios_removidos_malha": 14,
 "faces": 54227,
 "faces_sem_linha": 870,
 "agua": 746,
 "h3": 1943,
 "area_sem_cobertura_m2": 0.0,
 "rpa_por_campo_conferida": true,
 "politica": "união das duas fontes; não recortar bairros; pontos nas divergências sinalizados"
}


In [6]:
import geopandas as gpd
grade = gpd.read_parquet(SILVER / "grade_h3.parquet")
faces = pd.read_parquet(SILVER / "face_quadra.parquet")
rel = gpd.read_parquet(SILVER / "territorio_logradouro.parquet")
conferencia = pd.DataFrame({
    "obtido": [estatisticas["bairros"], estatisticas["rios_removidos_malha"], len(faces), round(100 * faces.coleta_manual.mean(), 1),
               round(100 * faces.limpeza_mensal_ou_sem.mean(), 1), round(100 * faces.sem_galeria.mean(), 1), round(100 * faces.v0.isna().mean(), 1), len(grade)],
    "referência (pastas 09 e 10)": [94, 14, 54227, 24.3, 27.9, 69.2, 42.5, "1.742 (outro método: só centro dentro do contorno)"],
}, index=["bairros", "rios cadastrados como rua", "faces", "% faces com coleta manual", "% limpeza mensal ou nenhuma", "% sem galeria", "% V0 ausente", "hexágonos H3 res. 9"])
conferencia

                              obtido                         referência (pastas 09 e 10)
bairros                         94.0                                                  94
rios cadastrados como rua       14.0                                                  14
faces                        54227.0                                               54227
% faces com coleta manual       24.4                                                24.3
% limpeza mensal ou nenhuma     27.9                                                27.9
% sem galeria                   69.2                                                69.2
% V0 ausente                    42.7                                                42.5
hexágonos H3 res. 9           1943.0  1.742 (outro método: só centro dentro do contorno)


In [7]:
assert estatisticas["bairros"] == 94 and estatisticas["rios_removidos_malha"] == 14
assert estatisticas["area_sem_cobertura_m2"] < 0.01, "a grade precisa cobrir todo o domínio"
assert grade.h3.is_unique and faces.dsqf.is_unique
assert rel.crs is not None
print("Relações rua-bairro sem linha utilizável:", estatisticas["relacoes_sem_linha_util"], "de", estatisticas["relacoes_rua_bairro"])
print("Pares rua-bairro com nome ambíguo:", estatisticas["relacoes_ambiguas"], "| faces sem linha de rua:", estatisticas["faces_sem_linha"])
print("Hexágonos em borda divergente:", int(grade.borda_divergente.sum()), "| sem bairro (só IBGE):", int(grade.bairro_codigo.isna().sum()))

Relações rua-bairro sem linha utilizável: 223 de 11921
Pares rua-bairro com nome ambíguo: 127 | faces sem linha de rua: 870
Hexágonos em borda divergente: 315 | sem bairro (só IBGE): 4


## Limites

- A nota metodológica da malha IBGE continua sem leitura integral (403); a comparação é técnica, não legal.
- `area_urbanizada_m2` não é estimada: não há camada de área construída. Onde for preciso, usa-se o proxy "tem via e faces cadastradas", sempre nomeado como proxy.
- As faces não têm data: o plano de serviço pode estar desatualizado.